In [1]:
!pip install -U datasets fsspec transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency

## Dataset

In [2]:
##from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [3]:
dataset = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(500))

## Tokenization

In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
def tokenize(example):
  return tokenizer(example["text"],padding="max_length",truncation=True,max_length=256)

In [8]:
def preprocess(ds):
  ds = ds.map(tokenize,batched=True,remove_columns=['text'])
  ds =ds.rename_column("label","labels")
  ds.set_format(type="torch",columns=["input_ids","attention_mask","labels"])
  return ds

In [9]:
train_dataset = preprocess(train_dataset)
test_dataset = preprocess(test_dataset)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Finetuning

In [ ]:
for param in model.bert.parameters():
  param.requires_grad = False

for layer in model.bert.encoder.layer[-2:]:
  for param in layer.parameters():
    param.requires_grad = True

In [11]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    logging_steps=10,
    weight_decay=0.01,
    report_to="none",
    learning_rate=2e-5
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset
)

In [13]:
trainer.train()

Step,Training Loss
10,0.263112
20,0.054620
30,0.015476
40,0.006932
50,0.004341
60,0.003211
70,0.002699
80,0.002271
90,0.002096
100,0.001942


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.028894582219421865, metrics={'train_runtime': 64.1085, 'train_samples_per_second': 15.599, 'train_steps_per_second': 1.95, 'total_flos': 131555527680000.0, 'train_loss': 0.028894582219421865, 'epoch': 1.0})

In [14]:
for layer in model.bert.encoder.layer:
  print(layer)

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)
BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (qu

In [18]:
!tensorboard --logdir=./logs


/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:292: SyntaxWarning: invalid escape sequence '\s'
  "[`\000-\040\177-\240\s]+",
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:339: SyntaxWarning: invalid escape sequence '\s'
  style = re.compile('url\s*\(\s*[^\s)]+?\s*\)\s*').sub(' ', style)
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:354: SyntaxWarning: invalid escape sequence '\s'
  if not re.match("^\s*([-\w]+\s*:[^:;]*(;\s*|$))*$", style):
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:358: SyntaxWarning: invalid escape sequence '\w'
  for prop, value in re.findall('([-\w]+)\s*:\s*([^:;]*)', style):
2026-02-16 15:01:37.696244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771254098.004982    5586 cuda_dnn.

## Save Model

In [17]:
trainer.save_model("./bert-finetuned-imdb")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
tokenizer.save_pretrained("./bert-finetuned-imdb")

('./bert-finetuned-imdb/tokenizer_config.json',
 './bert-finetuned-imdb/tokenizer.json')

## Evaluation

In [20]:
metrics = trainer.evaluate()

In [21]:
print(metrics)

{'eval_loss': 0.0014263105113059282, 'eval_runtime': 9.0063, 'eval_samples_per_second': 55.516, 'eval_steps_per_second': 6.995, 'epoch': 1.0}


In [22]:
tokenizer = BertTokenizer.from_pretrained('/content/bert-finetuned-imdb')
model = BertForSequenceClassification.from_pretrained("/content/bert-finetuned-imdb")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Testing

In [23]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model = model,
                      tokenizer = tokenizer)

In [24]:
text = "This movie was amazing and i loved the acting!"
result = classifier(text)

In [26]:
print(result)

[{'label': 'LABEL_0', 'score': 0.9969382286071777}]


## Push to HuggingFace

In [36]:
from huggingface_hub import notebook_login
notebook_login()

In [37]:
from huggingface_hub import whoami
print (whoami())

{'type': 'user', 'id': '68a6ac5133e6a6eabdcc3d6b', 'name': 'SANGAMESWAR', 'fullname': 'P K SANGAMESWAR', 'email': 'sanguchachu@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1772323200, 'isPro': False, 'avatarUrl': '/avatars/bc0fc3b104554d824f3b40d87c06249a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'hfcourse', 'role': 'write', 'createdAt': '2026-02-15T02:59:51.072Z'}}}


In [38]:
tokenizer.push_to_hub("SANGAMESWAR/my-bert-imdb2")

CommitInfo(commit_url='https://huggingface.co/SANGAMESWAR/my-bert-imdb2/commit/85ae60396ecb6f83168ecb98621074b220a1cab9', commit_message='Upload tokenizer', commit_description='', oid='85ae60396ecb6f83168ecb98621074b220a1cab9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SANGAMESWAR/my-bert-imdb2', endpoint='https://huggingface.co', repo_type='model', repo_id='SANGAMESWAR/my-bert-imdb2'), pr_revision=None, pr_num=None)

In [39]:
trainer.push_to_hub("SANGAMESWAR/my-bert-imdb2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/SANGAMESWAR/results/commit/b33a2a76f0d867aa4a40621de44fcf532e581f1f', commit_message='SANGAMESWAR/my-bert-imdb2', commit_description='', oid='b33a2a76f0d867aa4a40621de44fcf532e581f1f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SANGAMESWAR/results', endpoint='https://huggingface.co', repo_type='model', repo_id='SANGAMESWAR/results'), pr_revision=None, pr_num=None)